<a href="https://colab.research.google.com/github/ASaragga/GRF/blob/main/SimMonteCarlo01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import Pkg; Pkg.add("Distributions")
using Distributions, Statistics

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.10/Project.toml`
  No Changes to `~/.julia/environments/v1.10/Manifest.toml`


In [ ]:
# Formula de Black-Scholes para uma Call Option Europeia
function black_scholes_call(S, K, r, σ, T)
    d1 = (log(S / K) + (r + 0.5 * σ^2) * T) / (σ * sqrt(T))
    d2 = d1 - σ * sqrt(T)
    C = S * cdf(Normal(0,1), d1) - K * exp(-r * T) * cdf(Normal(0,1), d2)
    return C
end

# Parametros
S0 = 9.0    # Preço Spot
K = 9.2     # Preço Exercício
r = 0.03    # Taxa de Juro s/ Risco anual
σ = 0.20    # Volatilidade anual
T = 3/12    # Tempo até ao Vencimento em anos
N_sim = 100000  # Número de Simulações de Monte Carlo (cem mil)

100000

In [ ]:
# 1. Calculamos o preço inicial (hoje) da opção
C0 = black_scholes_call(S0, K, r, σ, T)

0.2996661269412142

In [ ]:
# 2. Simulamos o futuro valor spot da opção, usando um Movimento Browniano Geométrico
Δt1 = 1/252   # Um dia
Δt10 = 10/252 # Dez dias

distribution = Normal(0, 1)
Z1 = rand(distribution, N_sim)
Z10 = rand(distribution, N_sim)

S1 = S0 * exp.((r - 0.5 * σ^2) * Δt1 .+ σ * sqrt(Δt1) * Z1)
S10 = S0 * exp.((r - 0.5 * σ^2) * Δt10 .+ σ * sqrt(Δt10) * Z10)

100000-element Vector{Float64}:
 8.52399088197212
 9.480541306057473
 8.996307664059408
 9.254981742488846
 8.874847781843584
 8.682796164183236
 9.001869160337964
 8.462576002435084
 8.65150368704727
 9.06187514902013
 9.074076383384025
 8.715751083960601
 9.026745671515764
 ⋮
 8.747096506838028
 9.418224415723934
 8.845386159513758
 9.087657243278814
 9.19309534170821
 9.73185803563204
 8.23071541214085
 9.602740098539588
 8.53725481795312
 8.616040535001172
 9.450792465704186
 8.99971745090881

In [ ]:
# 3. Usando o Black-Scholes, calculamos os futuros preços da opção (1 e 10 dias no futuro).
C1 = black_scholes_call.(S1, K, r, σ, T - Δt1)
C10 = black_scholes_call.(S10, K, r, σ, T - Δt10)

100000-element Vector{Float64}:
 0.10372297666798524
 0.5361745134790743
 0.26390563528293187
 0.3958955906423425
 0.21295439523568271
 0.14646189898663442
 0.2664073830675653
 0.08992648939534553
 0.13719792794931562
 0.2943437481980924
 0.3002354099779909
 0.15668227536412793
 0.2777792637190277
 ⋮
 0.16685261635334347
 0.49518980418640535
 0.20164968625553836
 0.3068771179880292
 0.361432650714657
 0.7171709809026501
 0.04987531571085091
 0.6211719724664198
 0.10689545453440474
 0.12720877257779728
 0.5164039813134274
 0.2654377128686436

In [ ]:
# 4. Calculamos a distribuição de Ganhos & Perdas
L1 = C1 .- C0
PL10 = C10 .- C0

100000-element Vector{Float64}:
 -0.19594315027322895
  0.2365083865378601
 -0.03576049165828232
  0.09622946370112828
 -0.08671173170553148
 -0.15320422795457977
 -0.03325874387364891
 -0.20973963754586866
 -0.16246819899189857
 -0.005322378743121803
  0.0005692830367767243
 -0.14298385157708626
 -0.02188686322218647
  ⋮
 -0.13281351058787072
  0.19552367724519115
 -0.09801644068567583
  0.007210991046814996
  0.06176652377344283
  0.4175048539614359
 -0.24979081123036329
  0.32150584552520556
 -0.19277067240680945
 -0.17245735436341691
  0.21673785437221316
 -0.0342284140725706

In [ ]:
# 5. Calculamos o VaR e ETL ao nível de significância de 5%
VaR_1d = -quantile(PL1, 0.05)
ETL_1d = -mean(filter(x -> x < -VaR_1d, PL1))

VaR_10d = -quantile(PL10, 0.05)
ETL_10d = -mean(filter(x -> x < -VaR_10d, PL10))

println("Option Price Today = ", C0)
println("1-Day VaR (5%) = ",  VaR_1d)
println("1-Day ETL (5%) = ",  ETL_1d)
println("10-Day VaR (5%) = ", VaR_10d)
println("10-Day ETL (5%) = ", ETL_10d)

Option Price Today = 0.2996661269412142
1-Day VaR (5%) = 0.08068712265687583
1-Day ETL (5%) = 0.09745089007086796
10-Day VaR (5%) = 0.21611068305556724
10-Day ETL (5%) = 0.23865145475662308
